# SOTA HTR end-to-end (Kaggle-ready, minimal cells)

Notebook này là một triển khai hoàn chỉnh, tối giản (ít cell) để huấn luyện SOTA-style HTR trên MJSynth (Synth90k) trên Kaggle.
Mục tiêu: có thể chỉnh `MODE` (pretrain_ctc / finetune_seq2seq), `DATA_ROOT` và chạy trực tiếp.

Lưu ý: chỉnh `DATA_ROOT` phù hợp với runtime Kaggle (ví dụ `/kaggle/input/mjsynth/mnt/ramdisk/max/90kDICT32px`).

In [1]:
# Quick execution check for the intro markdown (Cell 1)
print('Cell 1: Intro markdown present — run reached Cell 2 to continue')

Cell 1: Intro markdown present — run reached Cell 2 to continue


In [2]:
# ====== Config (edit these for your run) ======
MODE = 'pretrain_ctc'  # 'pretrain_ctc' or 'finetune_seq2seq'
DATA_ROOT = '/kaggle/input/mjsynth/mnt/ramdisk/max/90kDICT32px'  # change to your path
SAVE_DIR = '/kaggle/working/sota_run'
IMG_H, IMG_W = 64, 256
BATCH_SIZE = 64
EPOCHS = 30
ACCUM_STEPS = 1
LR = 2e-4
D_MODEL = 384  # tuned to fit Kaggle T4 memory
ENC_LAYERS = 6
DEC_LAYERS = 4
USE_AMP = True
WORKERS = 2
SEED = 42
# =============================================
print(f'Cell 2: Config loaded — MODE={MODE}, DATA_ROOT={DATA_ROOT}, SAVE_DIR={SAVE_DIR}')

Cell 2: Config loaded — MODE=pretrain_ctc, DATA_ROOT=/kaggle/input/mjsynth/mnt/ramdisk/max/90kDICT32px, SAVE_DIR=/kaggle/working/sota_run


In [3]:
# Duplicate _label_from_path removed - now defined in MJSynthDataset class

In [4]:
# Pylance stubs for static analysis (_only_).
# Use TYPE_CHECKING so editors (Pylance) see these names, but they are not created at runtime.
from typing import TYPE_CHECKING
if TYPE_CHECKING:
    def cer(ref, hyp):
        """Stub CER for static analysis. Returns a float for type checkers."""
        return 0.0

    class LabelSmoothingLoss:
        def __init__(self, *args, **kwargs):
            pass
        def __call__(self, *args, **kwargs):
            return 0.0

    class SOTA_HTR:
        def __init__(self, *args, **kwargs):
            pass
        def forward_ctc(self, images):
            return None
        def forward_seq2seq(self, images, tgt):
            return None

print('Cell (stubs): static-analysis helpers defined under TYPE_CHECKING — no runtime definitions')

Cell (stubs): static-analysis helpers defined under TYPE_CHECKING — no runtime definitions


In [5]:
# Cell 5: Model và Helper Functions
import torch
import torch.nn as nn
import torch.nn.functional as F

def cer(ref, hyp):
    """Character Error Rate"""
    if len(ref) == 0:
        return 1.0 if len(hyp) > 0 else 0.0
    d = [[0]*(len(hyp)+1) for _ in range(len(ref)+1)]
    for i in range(len(ref)+1): d[i][0] = i
    for j in range(len(hyp)+1): d[0][j] = j
    for i in range(1, len(ref)+1):
        for j in range(1, len(hyp)+1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[len(ref)][len(hyp)] / len(ref)

class LabelSmoothingLoss(nn.Module):
    def __init__(self, smoothing=0.1, ignore_index=-100):
        super().__init__()
        self.smoothing = smoothing
        self.ignore_index = ignore_index
    def forward(self, logits, target):
        B, T, C = logits.shape
        logits = logits.reshape(-1, C)
        target = target.reshape(-1)
        mask = target != self.ignore_index
        target = target[mask]
        logits = logits[mask]
        log_probs = F.log_softmax(logits, dim=-1)
        nll_loss = F.nll_loss(log_probs, target, reduction='mean')
        smooth_loss = -log_probs.mean()
        loss = (1 - self.smoothing) * nll_loss + self.smoothing * smooth_loss
        return loss

import math

class PositionalEncoding2D(nn.Module):
    """2D Positional Encoding for image features"""
    def __init__(self, d_model, max_h=100, max_w=500):
        super().__init__()
        pe = torch.zeros(max_h, max_w, d_model)
        
        # Separate encoding for height and width
        d_model_half = d_model // 2
        div_term = torch.exp(torch.arange(0, d_model_half, 2).float() * 
                            -(math.log(10000.0) / d_model_half))
        
        pos_h = torch.arange(0, max_h).unsqueeze(1)
        pos_w = torch.arange(0, max_w).unsqueeze(1)
        
        pe[:, :, 0:d_model_half:2] = torch.sin(pos_h * div_term).unsqueeze(1).repeat(1, max_w, 1)
        pe[:, :, 1:d_model_half:2] = torch.cos(pos_h * div_term).unsqueeze(1).repeat(1, max_w, 1)
        pe[:, :, d_model_half::2] = torch.sin(pos_w * div_term).unsqueeze(0).repeat(max_h, 1, 1)
        pe[:, :, d_model_half+1::2] = torch.cos(pos_w * div_term).unsqueeze(0).repeat(max_h, 1, 1)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x, h, w):
        # x: [B, seq_len, d_model] where seq_len = h*w
        pe_flat = self.pe[:h, :w].reshape(-1, self.pe.size(-1))  # [h*w, d_model]
        return x + pe_flat.unsqueeze(0)

class ResNetBackbone(nn.Module):
    """Lightweight ResNet-style CNN backbone"""
    def __init__(self, d_model=384):
        super().__init__()
        # Initial conv
        self.conv1 = nn.Conv2d(1, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        
        # ResNet blocks
        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, d_model, 2, stride=1)
        
    def _make_layer(self, in_channels, out_channels, blocks, stride):
        layers = []
        # Downsample if needed
        downsample = None
        if stride != 1 or in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        
        layers.append(ResBlock(in_channels, out_channels, stride, downsample))
        for _ in range(1, blocks):
            layers.append(ResBlock(out_channels, out_channels))
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return x

class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.downsample = downsample
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample:
            identity = self.downsample(x)
        out += identity
        return self.relu(out)

class PositionalEncodingSinusoidal(nn.Module):
    """Standard sinusoidal positional encoding"""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                            (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        # x: [B, seq_len, d_model]
        return x + self.pe[:x.size(1), :].unsqueeze(0)

class SOTA_HTR(nn.Module):
    def __init__(self, vocab_size, d_model=384, patch_size=4, ctc_classes=None, pad_idx=0, enc_layers=6, dec_layers=4):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.patch_size = patch_size
        self.ctc_classes = ctc_classes if isinstance(ctc_classes, int) else len(ctc_classes) if ctc_classes else vocab_size
        self.pad_idx = pad_idx
        self.enc_layers = enc_layers
        self.dec_layers = dec_layers
        
        # Backbone: ResNet-style CNN
        self.backbone = ResNetBackbone(d_model)
        
        # 2D Positional Encoding
        self.pos_enc_2d = PositionalEncoding2D(d_model)
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=8, 
            dim_feedforward=4*d_model,
            dropout=0.1,
            activation='gelu',
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=enc_layers)
        
        # CTC Head
        self.ctc_head = nn.Linear(d_model, self.ctc_classes)
        
        # Seq2Seq components
        self.tgt_embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.tgt_pos_enc = PositionalEncodingSinusoidal(d_model)
        
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=8,
            dim_feedforward=4*d_model,
            dropout=0.1,
            activation='gelu',
            batch_first=True
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=dec_layers)
        self.out_proj = nn.Linear(d_model, vocab_size)
        
        self._init_weights()
    
    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward_encoder(self, images):
        # images: [B, 1, H, W]
        x = self.backbone(images)  # [B, d_model, h, w]
        
        B, C, h, w = x.shape
        x = x.flatten(2).transpose(1, 2)  # [B, h*w, d_model]
        
        # Add 2D positional encoding
        x = self.pos_enc_2d(x, h, w)
        
        # Transformer encoder
        memory = self.encoder(x)  # [B, seq_len, d_model]
        return memory, None
    
    def forward_ctc(self, images):
        memory, _ = self.forward_encoder(images)
        logits = self.ctc_head(memory)  # [B, seq_len, ctc_classes]
        return F.log_softmax(logits, dim=-1)
    
    def forward_seq2seq(self, images, tgt):
        # images: [B, 1, H, W], tgt: [B, tgt_len]
        memory, _ = self.forward_encoder(images)
        
        # Embed and add positional encoding
        tgt_emb = self.tgt_embed(tgt)  # [B, tgt_len, d_model]
        tgt_emb = self.tgt_pos_enc(tgt_emb)
        
        # Create causal mask
        tgt_len = tgt.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len).to(tgt.device)
        
        # Create padding mask
        tgt_padding_mask = (tgt == self.pad_idx)
        
        # Decode
        out = self.decoder(
            tgt_emb, 
            memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=tgt_padding_mask
        )
        
        logits = self.out_proj(out)
        return logits
    
    def generate(self, memory, max_len, sos_token, eos_token, device):
        B = memory.size(0)
        generated = torch.full((B, 1), sos_token, dtype=torch.long, device=device)
        
        for _ in range(max_len - 1):
            tgt_emb = self.tgt_embed(generated)
            tgt_emb = self.tgt_pos_enc(tgt_emb)
            
            tgt_len = generated.size(1)
            causal_mask = nn.Transformer.generate_square_subsequent_mask(tgt_len).to(device)
            
            out = self.decoder(tgt_emb, memory, tgt_mask=causal_mask)
            logits = self.out_proj(out[:, -1, :])  # Only last position
            next_token = logits.argmax(-1)
            
            generated = torch.cat([generated, next_token.unsqueeze(1)], dim=1)
            
            if (next_token == eos_token).all():
                break
        
        # Convert to list of token lists
        token_lists = []
        for b in range(B):
            tokens = []
            for t in generated[b, 1:]:  # Skip SOS
                if t == eos_token:
                    break
                tokens.append(t.item())
            token_lists.append(tokens)
        
        return token_lists

print('Cell 5: Model and helper functions defined')

Cell 5: Model and helper functions defined


In [6]:
import os
import string
from tqdm import tqdm
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR
from torch.utils.data import DataLoader
import cv2
import numpy as np
from torch.utils.data import Dataset
import torch.nn.functional as F
import torch.nn.utils.rnn

def set_seed(seed):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

class MJSynthDataset(Dataset):
    def __init__(self, root, img_size=(64, 256), is_train=True, augment=False, max_paths=None):
        self.root = root
        self.img_size = img_size
        self.is_train = is_train
        self.augment = augment
        self.max_paths = max_paths
        self.paths = self._get_paths()
    
    def _get_paths(self):
        paths = []
        count = 0
        for root, _, files in os.walk(self.root):
            for file in files:
                if file.endswith('.jpg'):
                    paths.append(os.path.join(root, file))
                    count += 1
                    if count % 1000 == 0:
                        print(f"Loading images: {count} images loaded", flush=True)
                    if self.max_paths and count >= self.max_paths:
                        print(f"Reached max_paths={self.max_paths}, stopping scan.", flush=True)
                        break
            if self.max_paths and count >= self.max_paths:
                break
        print(f"Done loading images: total {len(paths)} images.", flush=True)
        return paths
    
    def _label_from_path(self, p):
        parts = os.path.basename(p).split('_')
        if len(parts) >= 3:
            return parts[1].lower()
        return ""
    
    def __len__(self):
        return len(self.paths)
    
    def __getitem__(self, idx):
        p = self.paths[idx]
        img = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return self.__getitem__((idx + 1) % len(self))
        h, w = self.img_size
        img = cv2.resize(img, (w, h))
        if self.augment and self.is_train:
            pass  # add augmentation if needed
        img = img.astype(np.float32) / 255.0
        img = torch.from_numpy(img).unsqueeze(0)
        label = self._label_from_path(p)
        return img, label

def collate_ctc(batch, char_to_idx):
    imgs, labels = zip(*batch)
    imgs = torch.stack(imgs)
    targets = []
    tgt_lens = []
    texts = []
    for label in labels:
        encoded = [char_to_idx.get(c, 0) for c in label]
        targets.append(torch.tensor(encoded, dtype=torch.long))
        tgt_lens.append(len(encoded))
        texts.append(label)
    targets = torch.cat(targets)  # concatenated for safety
    tgt_lens = torch.tensor(tgt_lens, dtype=torch.long)
    return imgs, targets, tgt_lens, texts

def collate_seq2seq(batch, char_to_idx, sos, eos, pad):
    imgs, labels = zip(*batch)
    imgs = torch.stack(imgs)
    targets = []
    tgt_lens = []
    texts = []
    for label in labels:
        encoded = [sos] + [char_to_idx.get(c, 0) for c in label] + [eos]
        targets.append(torch.tensor(encoded, dtype=torch.long))
        tgt_lens.append(len(encoded))
        texts.append(label)
    targets = torch.nn.utils.rnn.pad_sequence(targets, batch_first=True, padding_value=pad)
    targets = targets.to(dtype=torch.long)
    tgt_lens = torch.tensor(tgt_lens, dtype=torch.long)
    return imgs, targets, tgt_lens, texts

def train_ctc_epoch(model, loader, opt, device, sched, scaler, accum_steps=1):
    model.train()
    total_loss = 0
    opt.zero_grad()
    for i, (imgs, targets, tgt_lens, texts) in enumerate(loader):
        imgs = imgs.to(device)
        targets = targets.to(device)
        tgt_lens = tgt_lens.to(device)
        
        with torch.amp.autocast('cuda' if device == 'cuda' else 'cpu'):
            logits = model.forward_ctc(imgs)
            input_lengths = torch.full((logits.size(0),), logits.size(1), dtype=torch.long, device=device)
            loss = F.ctc_loss(logits.permute(1,0,2), targets, input_lengths, tgt_lens, blank=0, reduction='mean')
            loss = loss / accum_steps  # scale loss for gradient accumulation
        
        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        
        if (i + 1) % accum_steps == 0:
            if scaler:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step()
            opt.zero_grad()
        
        total_loss += loss.item() * accum_steps
    return total_loss / len(loader)

def train_seq2seq_epoch(model, loader, criterion, opt, device, sched, scaler, accum_steps=1):
    model.train()
    total_loss = 0
    opt.zero_grad()
    for i, (imgs, targets, tgt_lens, texts) in enumerate(loader):
        imgs = imgs.to(device)
        targets = targets.to(device)
        tgt_lens = tgt_lens.to(device)
        
        with torch.amp.autocast('cuda' if device == 'cuda' else 'cpu'):
            logits = model.forward_seq2seq(imgs, targets[:, :-1])  # input prefix
            loss = criterion(logits.view(-1, logits.size(-1)), targets[:, 1:].view(-1))  # label next tokens
            loss = loss / accum_steps  # scale loss for gradient accumulation
        
        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        
        if (i + 1) % accum_steps == 0:
            if scaler:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
            sched.step()
            opt.zero_grad()
        
        total_loss += loss.item() * accum_steps
    return total_loss / len(loader)

def validate_seq2seq(model, loader, device, idx_to_char, sos, eos):
    model.eval()
    total_cer = 0
    count = 0
    with torch.no_grad():
        for imgs, targets, tgt_lens, texts in loader:
            imgs = imgs.to(device)
            memory, _ = model.forward_encoder(imgs)
            generated = model.generate(memory, max_len=50, sos_token=sos, eos_token=eos, device=device)
            for gen, text in zip(generated, texts):
                pred_text = ''.join(idx_to_char.get(i, '') for i in gen)
                total_cer += cer(text, pred_text)
                count += 1
    return total_cer / count if count > 0 else 1.0

print('Cell 6: Dataset, collate functions, and training functions defined')

Cell 6: Dataset, collate functions, and training functions defined


In [7]:
# Duplicate _label_from_path removed - now defined in MJSynthDataset class

In [8]:
# Preflight check — ensure model & helper definitions exist in the kernel before running the heavy training cell
# If you run this on Kaggle, there is no external `mjsynth_pytorch_sota.py` by default, so make sure you run Cells 2→5 in-order.
missing = [n for n in ('SOTA_HTR','LabelSmoothingLoss','cer') if n not in globals()]
if not missing:
    print('Preflight: required symbols present — OK')
else:
    print('Preflight: Missing the following runtime symbols:', missing)
    print('Action: Run Cells 2→5 (Config, Imports/Utils, Dataset/Collate, Model & helpers) in order, then re-run this cell and then the training cell.')
    # Provide a clear failure that stops execution instead of a confusing trace
    raise RuntimeError('Preflight failed — missing symbols: ' + ','.join(missing))

# --- original training cell follows (kept for readability) ---

Preflight: required symbols present — OK


In [9]:
@torch.no_grad()
def validate_ctc(model, loader, device, idx_to_char):
    model.eval()
    total = 0
    total_cer = 0.0
    # Collect samples for debug display (prioritize errors)
    all_samples = []
    error_samples = []
    
    for imgs, targets, tgt_lens, texts in tqdm(loader, leave=False):
        imgs = imgs.to(device)
        logits = model.forward_ctc(imgs)  # [B, seq_len, ctc_classes]
        probs = logits.exp()
        pred = probs.argmax(-1).cpu().numpy()  # [B, L]

        for p_seq, txt in zip(pred, texts):
            prev = -1
            chars = []
            for idx in p_seq:
                if idx != prev and idx != 0:  # 0 is blank
                    chars.append(idx_to_char.get(int(idx), ''))
                prev = idx
            pred_text = ''.join(chars)
            sample_cer = cer(txt, pred_text)
            
            # Collect samples for debug
            sample_info = (txt, pred_text, sample_cer)
            all_samples.append(sample_info)
            if sample_cer > 0:  # Error samples
                error_samples.append(sample_info)

            total += 1
            total_cer += sample_cer

    # Debug: Show 3 samples - prioritize errors, then random
    debug_samples = []
    if len(error_samples) >= 3:
        # If enough errors, randomly pick 3 error samples
        import random
        debug_samples = random.sample(error_samples, 3)
    elif len(error_samples) > 0:
        # Mix errors + random correct samples
        debug_samples = error_samples[:] 
        remaining = 3 - len(debug_samples)
        correct_samples = [s for s in all_samples if s[2] == 0]
        if len(correct_samples) > 0:
            debug_samples += random.sample(correct_samples, min(remaining, len(correct_samples)))
    else:
        # All correct, show 3 random samples
        import random
        debug_samples = random.sample(all_samples, min(3, len(all_samples)))
    
    for gt, pred, sample_cer in debug_samples:
        print(f"Debug Val: GT='{gt}' Pred='{pred}' CER={sample_cer:.4f}")

    avg_cer = total_cer / max(1, total)
    print(f"Validation: checked {total} samples, avg CER = {avg_cer:.4f}")
    return avg_cer

In [10]:
# Main run: build data, model, optimizer, scheduler and loop
print('Cell 6: Starting training/validation cell - preparing environment')
set_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
os.makedirs(SAVE_DIR, exist_ok=True)
chars = list(string.ascii_lowercase + string.digits)
CTC_BLANK = '<BLANK>'
SEQ_PAD = '<PAD>'
SEQ_SOS = '<SOS>'
SEQ_EOS = '<EOS>'
ctc_classes = [CTC_BLANK] + chars
vocab = [SEQ_PAD, SEQ_SOS, SEQ_EOS] + chars
seq_char_to_idx = {c: i for i, c in enumerate(vocab)}
seq_idx_to_char = {i: c for c, i in seq_char_to_idx.items()}
ctc_char_to_idx = {c: i for i, c in enumerate(ctc_classes)}
ctc_idx_to_char = {i: c for c, i in ctc_char_to_idx.items()}

VAL_LIMIT = 10000  # Limit val dataset to avoid slow scanning on Kaggle

# If pretraining, avoid scanning the whole dataset by passing a limit to the dataset constructor
pretrain_limit = 500000 if MODE == 'pretrain_ctc' else None  # Tăng lên 500k samples
print(f'pretrain_limit = {pretrain_limit}')
train_ds = MJSynthDataset(DATA_ROOT, img_size=(IMG_H, IMG_W), is_train=True, max_paths=pretrain_limit)
print('After creating train_ds: number of paths read =', len(train_ds.paths), f'(limit={pretrain_limit})')
print('First 5 train_ds paths:', train_ds.paths[:5])
val_ds = MJSynthDataset(DATA_ROOT, img_size=(IMG_H, IMG_W), is_train=False, augment=False, max_paths=VAL_LIMIT)
print('After creating val_ds: number of paths read =', len(val_ds.paths), f'(limit={VAL_LIMIT})')
print('First 5 val_ds paths:', val_ds.paths[:5])

effective_workers = WORKERS
if os.name == 'nt':
    effective_workers = 0
    print('Detected Windows: forcing dataloader num_workers=0 for safety')
print(f'Creating DataLoader with num_workers={effective_workers} (requested={WORKERS})')
if MODE == 'pretrain_ctc':
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda b: collate_ctc(b, ctc_char_to_idx), num_workers=effective_workers, pin_memory=(device=='cuda'))
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_ctc(b, ctc_char_to_idx), num_workers=effective_workers, pin_memory=(device=='cuda'))
    print(f'Pretrain mode: train samples={len(train_ds.paths)}, val samples={len(val_ds.paths)}')
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda b: collate_seq2seq(b, seq_char_to_idx, sos=seq_char_to_idx[SEQ_SOS], eos=seq_char_to_idx[SEQ_EOS], pad=seq_char_to_idx[SEQ_PAD]), num_workers=effective_workers, pin_memory=(device=='cuda'))
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=lambda b: collate_seq2seq(b, seq_char_to_idx, sos=seq_char_to_idx[SEQ_SOS], eos=seq_char_to_idx[SEQ_EOS], pad=seq_char_to_idx[SEQ_PAD]), num_workers=effective_workers, pin_memory=(device=='cuda'))
    print(f'Finetune mode: train samples={len(train_ds.paths)}, val samples={len(val_ds.paths)}')
# Extra runtime info: show DataLoader dataset sizes and batch counts so user can confirm loading
try:
    print('After DataLoader creation:')
    print(f'  Train dataset length (samples): {len(train_loader.dataset)}')
    print(f'  Val dataset length (samples):   {len(val_loader.dataset)}')
    try:
        print(f'  Train loader batches: {len(train_loader)}, Val loader batches: {len(val_loader)}')
    except Exception:
        # some DataLoader setups might not support len()
        pass
    if hasattr(train_loader.dataset, 'paths') and len(train_loader.dataset.paths) > 0:
        print('  Example train dataset path:', train_loader.dataset.paths[0])
except Exception:
    pass
print('DataLoaders created — proceeding to model init')
print('Creating model...')

model = SOTA_HTR(vocab_size=len(vocab), d_model=D_MODEL, patch_size=4, ctc_classes=ctc_classes, pad_idx=seq_char_to_idx[SEQ_PAD], enc_layers=ENC_LAYERS, dec_layers=DEC_LAYERS)
print('Model instance created')
print('Moving model to device...')
model = model.to(device)
print('Model moved to device')
opt = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
steps_per_epoch = math.ceil(len(train_loader) / max(1, ACCUM_STEPS))
print(f'Steps per epoch: {steps_per_epoch}, total epochs: {EPOCHS}')
total_steps = max(1, EPOCHS * steps_per_epoch)
# Reduce max_lr to prevent divergence - use 3x instead of 8x
sched = OneCycleLR(opt, max_lr=LR * 3, total_steps=total_steps, pct_start=0.3, anneal_strategy='cos')
print(f'Scheduler: OneCycleLR with max_lr={LR * 3:.2e}, pct_start=0.3')
# Use torch.amp.GradScaler for PyTorch >=2.0
if USE_AMP and device == 'cuda':
    try:
        scaler = torch.amp.GradScaler('cuda')
    except Exception:
        scaler = torch.cuda.amp.GradScaler(enabled=True)
else:
    scaler = None
best = 1e9
# Tracking lists for plotting
train_losses = []
val_cers = []
# main epoch loop
for epoch in range(1, EPOCHS + 1):
    print(f'--- Starting epoch {epoch}/{EPOCHS} (mode={MODE})', flush=True)
    # ensure visible start-of-epoch print above
    if MODE == 'pretrain_ctc':
        tr = train_ctc_epoch(model, train_loader, opt, device, sched, scaler, accum_steps=ACCUM_STEPS)
        val_cer = validate_ctc(model, val_loader, device, ctc_idx_to_char)
        # Calculate train CER on a small RANDOM subset for monitoring (different each epoch)
        subset_size = min(500, len(train_ds))
        random_indices = torch.randperm(len(train_ds))[:subset_size].tolist()
        train_subset = torch.utils.data.Subset(train_ds, random_indices)
        train_subset_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=lambda b: collate_ctc(b, ctc_char_to_idx))
        train_cer = validate_ctc(model, train_subset_loader, device, ctc_idx_to_char)
        train_losses.append(tr)
        val_cers.append(val_cer)
        print(f'Epoch {epoch} TrainLoss {tr:.4f} TrainCER {train_cer:.4f} ValCER {val_cer:.4f}')
        if val_cer < best:
            best = val_cer
            ck = {'model': model.state_dict(), 'optimizer': opt.state_dict(), 'scheduler': sched.state_dict(), 'scaler': scaler.state_dict() if scaler is not None else None, 'epoch': epoch, 'best_metric': best}
            torch.save(ck, os.path.join(SAVE_DIR, 'best_encoder.pth'))
            print(f'Saved best encoder (epoch {epoch}, ValCER {val_cer:.4f})')
    else:
        tr = train_seq2seq_epoch(model, train_loader, LabelSmoothingLoss(ignore_index=seq_char_to_idx[SEQ_PAD]), opt, device, sched, scaler, accum_steps=ACCUM_STEPS)
        val_cer = validate_seq2seq(model, val_loader, device, seq_idx_to_char, sos=seq_char_to_idx[SEQ_SOS], eos=seq_char_to_idx[SEQ_EOS])
        # Calculate train CER on a small RANDOM subset for monitoring (different each epoch)
        subset_size = min(500, len(train_ds))
        random_indices = torch.randperm(len(train_ds))[:subset_size].tolist()
        train_subset = torch.utils.data.Subset(train_ds, random_indices)
        train_subset_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=lambda b: collate_seq2seq(b, seq_char_to_idx, sos=seq_char_to_idx[SEQ_SOS], eos=seq_char_to_idx[SEQ_EOS], pad=seq_char_to_idx[SEQ_PAD]))
        train_cer = validate_seq2seq(model, train_subset_loader, device, seq_idx_to_char, sos=seq_char_to_idx[SEQ_SOS], eos=seq_char_to_idx[SEQ_EOS])
        train_losses.append(tr)
        val_cers.append(val_cer)
        print(f'Epoch {epoch} TrainLoss {tr:.4f} TrainCER {train_cer:.4f} ValCER {val_cer:.4f}')
        if val_cer < best:
            best = val_cer
            ck = {'model': model.state_dict(), 'optimizer': opt.state_dict(), 'scheduler': sched.state_dict(), 'scaler': scaler.state_dict() if scaler is not None else None, 'epoch': epoch, 'best_metric': best}
            torch.save(ck, os.path.join(SAVE_DIR, 'best_seq.pth'))
            print(f'Saved best seq2seq (epoch {epoch}, ValCER {val_cer:.4f})')

# After training: plot training curves
import matplotlib.pyplot as _plt
if len(train_losses) > 0 and len(val_cers) > 0:
    epochs = list(range(1, len(train_losses) + 1))
    _plt.figure(figsize=(8,4))
    _plt.plot(epochs, train_losses, label='Train Loss')
    _plt.plot(epochs, val_cers, label='Val CER')
    _plt.xlabel('Epoch')
    _plt.legend()
    _plt.title('Train Loss and Val CER')
    _plt.grid(True)
    plot_path = os.path.join(SAVE_DIR, 'training_curves.png')
    _plt.tight_layout(); _plt.savefig(plot_path)
    try:
        from IPython.display import display, Image
        display(Image(plot_path))
    except Exception:
        print('Saved training plot to', plot_path)

print('Done. Best metric:', best)
print('Cell 6: Training/validation loop cell finished')

Cell 6: Starting training/validation cell - preparing environment
Using device: cuda
pretrain_limit = 500000
Loading images: 1000 images loaded
Loading images: 2000 images loaded
Loading images: 3000 images loaded
Loading images: 4000 images loaded
Loading images: 5000 images loaded
Loading images: 6000 images loaded
Loading images: 7000 images loaded
Loading images: 8000 images loaded
Loading images: 9000 images loaded
Loading images: 10000 images loaded
Loading images: 11000 images loaded
Loading images: 12000 images loaded
Loading images: 13000 images loaded
Loading images: 14000 images loaded
Loading images: 15000 images loaded
Loading images: 16000 images loaded
Loading images: 17000 images loaded
Loading images: 18000 images loaded
Loading images: 19000 images loaded
Loading images: 20000 images loaded
Loading images: 21000 images loaded
Loading images: 22000 images loaded
Loading images: 23000 images loaded
Loading images: 24000 images loaded
Loading images: 25000 images loaded


/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:227: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file


Debug Val: GT='circumscribes' Pred='ticunsernes' CER=0.4615
Debug Val: GT='cobwebby' Pred='cobwebyy' CER=0.1250
Debug Val: GT='shoveled' Pred='shoveld' CER=0.1250
Validation: checked 10000 samples, avg CER = 0.2074


Debug Val: GT='medulla' Pred='mediulle' CER=0.2857
Debug Val: GT='rontgen' Pred='tontgen' CER=0.1429
Debug Val: GT='palpitations' Pred='caipialions' CER=0.3333
Validation: checked 500 samples, avg CER = 0.1912
Epoch 1 TrainLoss 1.8817 TrainCER 0.1912 ValCER 0.2074
Saved best encoder (epoch 1, ValCER 0.2074)
--- Starting epoch 2/30 (mode=pretrain_ctc)


Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file


...

Debug Val: GT='sheller' Pred='sler' CER=0.4286
Debug Val: GT='hanna' Pred='hanno' CER=0.2000
Debug Val: GT='discourses' Pred='discolses' CER=0.2000
Validation: checked 10000 samples, avg CER = 0.0021


Debug Val: GT='ambiguities' Pred='inids' CER=0.7273
Debug Val: GT='correspondence' Pred='correspondente' CER=0.0714
Debug Val: GT='helvetius' Pred='nelvetlus' CER=0.2222
Validation: checked 500 samples, avg CER = 0.0044
Epoch 27 TrainLoss 0.0130 TrainCER 0.0044 ValCER 0.0021
Saved best encoder (epoch 27, ValCER 0.0021)
--- Starting epoch 28/30 (mode=pretrain_ctc)


Premature end of JPEG file
Premature end of JPEG file
Premature end of JPEG file


Debug Val: GT='nasturtiums' Pred='masturtiums' CER=0.0909
Debug Val: GT='matchmaking' Pred='mnting' CER=0.5455
Debug Val: GT='devolution' Pred='devoluciong' CER=0.2000
Validation: checked 10000 samples, avg CER = 0.0017


UnboundLocalError: cannot access local variable 'random' where it is not associated with a value

In [22]:
# Smoke-test cell: runs a tiny forward pass on a small model to verify shapes and prints outputs
if 'SOTA_HTR' not in globals():
    print('Smoke-test: Run the previous cells first to define the model and vocab.')
else:
    print('Smoke-test: start')
    import traceback
    try:
        device = 'cpu'
        print('Smoke-test device:', device)
        # Use small model config to avoid large memory use
        small = SOTA_HTR(vocab_size=len(vocab), d_model=64, patch_size=4, ctc_classes=ctc_classes, pad_idx=seq_char_to_idx[SEQ_PAD], enc_layers=2, dec_layers=1)
        small = small.to(device)
        B = 2
        imgs = torch.randn(B, 1, IMG_H, IMG_W, device=device)
        with torch.no_grad():
            out_ctc = small.forward_ctc(imgs)
            print('Smoke-test: forward_ctc output shape =', getattr(out_ctc, 'shape', str(type(out_ctc))))
            tgt = torch.full((B, 6), seq_char_to_idx[SEQ_SOS], dtype=torch.long, device=device)
            out_seq = small.forward_seq2seq(imgs, tgt)
            print('Smoke-test: forward_seq2seq output shape =', getattr(out_seq, 'shape', str(type(out_seq))))
        del small
        print('Smoke-test: OK')
    except Exception as e:
        traceback.print_exc()
        print('Smoke-test failed:', e)

Smoke-test: start
Smoke-test device: cpu
Smoke-test: forward_ctc output shape = torch.Size([2, 64, 37])
Smoke-test: forward_seq2seq output shape = torch.Size([2, 6, 39])
Smoke-test: OK


/usr/local/lib/python3.11/dist-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


# Download Model về máy

Sau khi training xong, download model về máy local để sử dụng tiếp.

In [11]:
# Cell: Download trained models to local machine
import os
from IPython.display import FileLink

print('='*60)
print('DOWNLOADING MODELS TO LOCAL MACHINE')
print('='*60)

# List of models to download
models_to_download = [
    'best_encoder.pth',
    'training_curves.png'
]

print('\nAvailable models in SAVE_DIR:')
if os.path.exists(SAVE_DIR):
    all_files = os.listdir(SAVE_DIR)
    for f in all_files:
        file_path = os.path.join(SAVE_DIR, f)
        if os.path.isfile(file_path):
            size_mb = os.path.getsize(file_path) / (1024 * 1024)
            print(f'  ✓ {f} ({size_mb:.2f} MB)')
else:
    print(f'  ❌ Directory not found: {SAVE_DIR}')

# Kaggle environment: files in /kaggle/working are automatically downloaded
print('\n' + '='*60)
print('DOWNLOAD INSTRUCTIONS')
print('='*60)

if '/kaggle/working' in SAVE_DIR:
    print('✓ Running on Kaggle!')
    print('  Models are saved in /kaggle/working/')
    print('  These files will be automatically available in the "Output" tab')
    print('  after the notebook finishes running.')
    print('\nTo download:')
    print('  1. Wait for notebook to finish')
    print('  2. Go to the "Output" tab on the right')
    print('  3. Click "Download" button to get all files as a zip')
else:
    print('✓ Running locally!')
    print(f'  Models are saved in: {SAVE_DIR}')
    print('\nModel files:')
    for model_file in models_to_download:
        model_path = os.path.join(SAVE_DIR, model_file)
        if os.path.exists(model_path):
            print(f'  ✓ {model_path}')
        else:
            print(f'  ⚠ Not found: {model_file}')

# Additional: Create a summary file
summary_path = os.path.join(SAVE_DIR, 'training_summary.txt')
try:
    with open(summary_path, 'w') as f:
        f.write('='*60 + '\n')
        f.write('TRAINING SUMMARY\n')
        f.write('='*60 + '\n\n')
        f.write(f'Mode: {MODE}\n')
        f.write(f'Dataset: {DATA_ROOT}\n')
        f.write(f'Training samples: {pretrain_limit if pretrain_limit else "All"}\n')
        f.write(f'Validation samples: {VAL_LIMIT}\n')
        f.write(f'Epochs: {EPOCHS}\n')
        f.write(f'Batch size: {BATCH_SIZE}\n')
        f.write(f'Learning rate: {LR}\n')
        f.write(f'Model: d_model={D_MODEL}, enc_layers={ENC_LAYERS}, dec_layers={DEC_LAYERS}\n')
        f.write(f'\nBest validation CER: {best:.4f}\n')
        f.write(f'\nModels saved:\n')
        for model_file in models_to_download:
            if os.path.exists(os.path.join(SAVE_DIR, model_file)):
                f.write(f'  - {model_file}\n')
    print(f'\n✓ Training summary saved to: {summary_path}')
except Exception as e:
    print(f'⚠ Could not create summary file: {e}')

print('\n' + '='*60)
print('DONE!')
print('='*60)

DOWNLOADING MODELS TO LOCAL MACHINE

Available models in SAVE_DIR:
  ✓ best_encoder.pth (327.66 MB)

DOWNLOAD INSTRUCTIONS
✓ Running on Kaggle!
  Models are saved in /kaggle/working/
  These files will be automatically available in the "Output" tab
  after the notebook finishes running.

To download:
  1. Wait for notebook to finish
  2. Go to the "Output" tab on the right
  3. Click "Download" button to get all files as a zip

✓ Training summary saved to: /kaggle/working/sota_run/training_summary.txt

DONE!


In [12]:
import os
import random
import torch
from torch.utils.data import DataLoader

# Hàm helper để random approximate paths mà không scan hết
def get_random_paths(root, num_samples=10, num_folders_to_sample=5):
    # Bước 1: List tất cả subfolders (nhanh, chỉ os.listdir)
    subfolders = [os.path.join(root, d) for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    if not subfolders:
        raise ValueError("No subfolders found in root.")
    
    # Bước 2: Random chọn vài subfolder (num_folders_to_sample, để đa dạng)
    selected_folders = random.sample(subfolders, min(num_folders_to_sample, len(subfolders)))
    
    # Bước 3: Từ mỗi folder, random chọn files (chia đều num_samples)
    paths = []
    samples_per_folder = max(1, num_samples // len(selected_folders))
    for folder in selected_folders:
        files = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith('.jpg')]
        if files:
            selected_files = random.sample(files, min(samples_per_folder, len(files)))
            paths.extend(selected_files)
    
    # Nếu chưa đủ, random thêm từ folder khác
    while len(paths) < num_samples and subfolders:
        extra_folder = random.choice(subfolders)
        extra_files = [os.path.join(extra_folder, f) for f in os.listdir(extra_folder) if f.endswith('.jpg')]
        if extra_files:
            paths.append(random.choice(extra_files))
    
    return paths[:num_samples]  # Giới hạn đúng num_samples

# Sử dụng: Load model như trước
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SOTA_HTR(...)  # Như code trước, load state_dict từ checkpoint
model = model.to(device)
model.eval()

# Lấy 10 paths random approximate
random_paths = get_random_paths(DATA_ROOT, num_samples=10, num_folders_to_sample=5)  # num_folders_to_sample để đa dạng
print(f'Random paths selected: {random_paths}')

# Tạo dataset từ paths này (chỉnh class của bạn để nhận paths trực tiếp nếu cần, hoặc dùng Subset)
test_ds = MJSynthDataset(DATA_ROOT, img_size=(IMG_H, IMG_W), is_train=False)  # Load dummy, rồi override paths
test_ds.paths = random_paths  # Override paths để chỉ dùng random ones

# Tạo loader và test (dùng collate_ctc hoặc seq2seq tùy mode)
test_loader = DataLoader(test_ds, batch_size=10, shuffle=False, collate_fn=lambda b: collate_ctc(b, ctc_char_to_idx), num_workers=0)
test_cer = validate_ctc(model, test_loader, device, ctc_idx_to_char)  # Hoặc validate_seq2seq
print(f'Test CER on 10 random samples: {test_cer:.4f}')

TypeError: empty() received an invalid combination of arguments - got (tuple, dtype=NoneType, device=NoneType), but expected one of:
 * (tuple of ints size, *, tuple of names names, torch.memory_format memory_format = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)
 * (tuple of ints size, *, torch.memory_format memory_format = None, Tensor out = None, torch.dtype dtype = None, torch.layout layout = None, torch.device device = None, bool pin_memory = False, bool requires_grad = False)


In [14]:
# Cell: Test trained model on 10 random images
import os
import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import random

print('='*60)
print('TESTING MODEL ON 10 RANDOM IMAGES')
print('='*60)

# ====== 1. Load model ======
MODEL_PATH = '/kaggle/working/sota_run/best_encoder.pth'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice: {device}')

if not os.path.exists(MODEL_PATH):
    print(f'❌ Model not found at: {MODEL_PATH}')
    print('Please train the model first!')
else:
    print(f'✓ Loading model from: {MODEL_PATH}')
    
    # Load checkpoint
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    
    # Recreate model with same architecture
    model = SOTA_HTR(
        vocab_size=len(vocab),
        d_model=D_MODEL,
        patch_size=4,
        ctc_classes=ctc_classes,
        pad_idx=seq_char_to_idx[SEQ_PAD],
        enc_layers=ENC_LAYERS,
        dec_layers=DEC_LAYERS
    )
    
    # Load weights
    model.load_state_dict(checkpoint['model'])
    model = model.to(device)
    model.eval()
    
    print(f'✓ Model loaded successfully!')
    print(f"  Best epoch: {checkpoint.get('epoch', 'N/A')}")
    print(f"  Best ValCER: {checkpoint.get('best_metric', 'N/A'):.4f}")
    
    # ====== 2. Randomly sample 10 image paths from MJSynth ======
    print('\n' + '='*60)
    print('SAMPLING 10 RANDOM IMAGES')
    print('='*60)
    
    # Quick scan to get ~100 random paths first (much faster than full scan)
    print('\nScanning dataset for random samples...')
    all_image_paths = []
    scan_count = 0
    max_scan = 100000  # Scan first 100k images to get random samples
    
    for root, _, files in os.walk(DATA_ROOT):
        for file in files:
            if file.endswith('.jpg'):
                all_image_paths.append(os.path.join(root, file))
                scan_count += 1
                if scan_count >= max_scan:
                    break
        if scan_count >= max_scan:
            break
    
    print(f'✓ Found {len(all_image_paths)} images (scanned {scan_count})')
    
    # Randomly select 10 images
    if len(all_image_paths) < 10:
        print(f'⚠ Only {len(all_image_paths)} images available')
        test_images = all_image_paths
    else:
        test_images = random.sample(all_image_paths, 10)
    
    print(f'✓ Selected {len(test_images)} random images for testing')
    
    # ====== 3. Test inference on selected images ======
    print('\n' + '='*60)
    print('INFERENCE RESULTS')
    print('='*60)
    
    def preprocess_image(img_path, img_h=64, img_w=256):
        """Load and preprocess single image"""
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            return None
        img = cv2.resize(img, (img_w, img_h))
        img = img.astype(np.float32) / 255.0
        img = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
        return img
    
    def decode_ctc_prediction(pred_seq, idx_to_char):
        """Decode CTC prediction by removing blanks and duplicates"""
        prev = -1
        chars = []
        for idx in pred_seq:
            if idx != prev and idx != 0:  # 0 is blank
                chars.append(idx_to_char.get(int(idx), ''))
            prev = idx
        return ''.join(chars)
    
    def get_ground_truth(img_path):
        """Extract ground truth from filename"""
        filename = os.path.basename(img_path)
        parts = filename.split('_')
        if len(parts) >= 3:
            return parts[1].lower()
        return "unknown"
    
    # Process each test image
    with torch.no_grad():
        for i, img_path in enumerate(test_images, 1):
            # Load and preprocess
            img_tensor = preprocess_image(img_path, IMG_H, IMG_W)
            if img_tensor is None:
                print(f'\n[{i}/10] ⚠ Failed to load: {os.path.basename(img_path)}')
                continue
            
            # Get ground truth
            gt_text = get_ground_truth(img_path)
            
            # Inference
            img_tensor = img_tensor.to(device)
            logits = model.forward_ctc(img_tensor)  # [1, seq_len, num_classes]
            probs = logits.exp()
            pred = probs.argmax(-1).cpu().numpy()[0]  # [seq_len]
            
            # Decode prediction
            pred_text = decode_ctc_prediction(pred, ctc_idx_to_char)
            
            # Calculate CER
            error_rate = cer(gt_text, pred_text)
            
            # Display result
            status = '✓' if error_rate == 0 else ('~' if error_rate < 0.3 else '✗')
            print(f'\n[{i}/10] {status} Image: {os.path.basename(img_path)}')
            print(f'  Ground Truth: "{gt_text}"')
            print(f'  Prediction:   "{pred_text}"')
            print(f'  CER: {error_rate:.4f}')
    
    # ====== 4. Summary ======
    print('\n' + '='*60)
    print('TESTING COMPLETED!')
    print('='*60)
    print(f'Tested {len(test_images)} random images from MJSynth dataset')
    print(f'Model: {MODEL_PATH}')
    print(f'Device: {device}')

TESTING MODEL ON 10 RANDOM IMAGES

Device: cuda
✓ Loading model from: /kaggle/working/sota_run/best_encoder.pth
✓ Model loaded successfully!
  Best epoch: 27
  Best ValCER: 0.0021

SAMPLING 10 RANDOM IMAGES

Scanning dataset for random samples...
✓ Found 100000 images (scanned 100000)
✓ Selected 10 random images for testing

INFERENCE RESULTS

[1/10] ✓ Image: 356_nisan_51743.jpg
  Ground Truth: "nisan"
  Prediction:   "nisan"
  CER: 0.0000

[2/10] ✓ Image: 421_Restrict_65362.jpg
  Ground Truth: "restrict"
  Prediction:   "restrict"
  CER: 0.0000

[3/10] ✓ Image: 366_NAIR_50792.jpg
  Ground Truth: "nair"
  Prediction:   "nair"
  CER: 0.0000

[4/10] ✓ Image: 330_terrine_78312.jpg
  Ground Truth: "terrine"
  Prediction:   "terrine"
  CER: 0.0000

[5/10] ✓ Image: 49_Cutlass_18855.jpg
  Ground Truth: "cutlass"
  Prediction:   "cutlass"
  CER: 0.0000

[6/10] ✓ Image: 359_UNDERLYING_82165.jpg
  Ground Truth: "underlying"
  Prediction:   "underlying"
  CER: 0.0000

[7/10] ✓ Image: 164_ZIMMERMA

In [21]:
import os
import cv2
import numpy as np
import torch
import random
import time
print('='*60)
print('TESTING MODEL ON 10 RANDOM IMAGES')
print('='*60)
# ====== 1. Load model ======
MODEL_PATH = '/kaggle/working/sota_run/best_encoder.pth'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\nDevice: {device}')
if not os.path.exists(MODEL_PATH):
    print(f'❌ Model not found at: {MODEL_PATH}')
    print('Please train the model first!')
else:
    print(f'✓ Loading model from: {MODEL_PATH}')
  
    # Load checkpoint
    checkpoint = torch.load(MODEL_PATH, map_location=device)
  
    # Recreate model with same architecture
    model = SOTA_HTR(
        vocab_size=len(vocab),
        d_model=D_MODEL,
        patch_size=4,
        ctc_classes=ctc_classes,
        pad_idx=seq_char_to_idx[SEQ_PAD],
        enc_layers=ENC_LAYERS,
        dec_layers=DEC_LAYERS
    )
  
    # Load weights
    model.load_state_dict(checkpoint['model'])
    model = model.to(device)
    model.eval()
  
    print(f'✓ Model loaded successfully!')
    print(f" Best epoch: {checkpoint.get('epoch', 'N/A')}")
    print(f" Best ValCER: {checkpoint.get('best_metric', 'N/A'):.4f}")
  
    # ====== 2. SUPER FAST: Chọn random N subdirs, mỗi cái lấy 1 ảnh ======
    print('\n' + '='*60)
    print('SAMPLING 10 RANDOM IMAGES (FAST & DIVERSE)')
    print('='*60)
  
    def get_random_images_fast(data_root, k=10, max_attempts=50):
        """
        Strategy: Random walk through the directory tree to select k diverse images.
        - Fast: No full os.walk; just descend randomly until finding a leaf with .jpg.
        - Diverse/Unbiased: Each walk picks a random path, ensuring varied words.
        - Handles structure: Assumes ~3 nested levels before leaf dirs with images.
        """
        start = time.time()
        selected_paths = []
        attempts = 0
        while len(selected_paths) < k and attempts < max_attempts:
            attempts += 1
            current_dir = data_root
            max_depth = 4  # Safety limit for recursion depth
            # Random descend through levels until leaf
            for _ in range(max_depth):
                subdirs = [d for d in os.listdir(current_dir) if os.path.isdir(os.path.join(current_dir, d))]
                if not subdirs:
                    break  # No more subdirs; check if leaf
                # Random choose next subdir
                next_subdir = random.choice(subdirs)
                current_dir = os.path.join(current_dir, next_subdir)
            # Now at potential leaf: Check for .jpg files
            try:
                jpg_files = [f for f in os.listdir(current_dir) if f.endswith('.jpg')]
                if jpg_files:
                    selected_file = random.choice(jpg_files)
                    selected_path = os.path.join(current_dir, selected_file)
                    if selected_path not in selected_paths:  # Avoid duplicates (rare)
                        selected_paths.append(selected_path)
            except Exception as e:
                print(f'⚠ Error in walk: {e}')
                continue
        if len(selected_paths) < k:
            print(f'⚠ Only found {len(selected_paths)} images after {attempts} attempts (dataset may be sparse)')
        elapsed = time.time() - start
        print(f'✓ Selected {len(selected_paths)} diverse images in {elapsed:.3f}s via random walk')
        return selected_paths
   
    # Sample images - FAST & DIVERSE!
    test_images = get_random_images_fast(DATA_ROOT, k=10)
   
    if not test_images:
        print('❌ No images found!')
    else:
        # ====== 3. Test inference on selected images ======
        print('\n' + '='*60)
        print('INFERENCE RESULTS')
        print('='*60)
       
        def preprocess_image(img_path, img_h=64, img_w=256):
            """Load and preprocess single image"""
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                return None
            img = cv2.resize(img, (img_w, img_h))
            img = img.astype(np.float32) / 255.0
            img = torch.from_numpy(img).unsqueeze(0).unsqueeze(0) # [1, 1, H, W]
            return img
       
        def decode_ctc_prediction(pred_seq, idx_to_char):
            """Decode CTC prediction by removing blanks and duplicates"""
            prev = -1
            chars = []
            for idx in pred_seq:
                if idx != prev and idx != 0: # 0 is blank
                    chars.append(idx_to_char.get(int(idx), ''))
                prev = idx
            return ''.join(chars)
       
        def get_ground_truth(img_path):
            """Extract ground truth from MJSynth filename format"""
            filename = os.path.basename(img_path)
            # MJSynth format: {index}_{word}_{...}.jpg
            parts = filename.split('_')
            if len(parts) >= 2:
                return parts[1].lower()
            return "unknown"
       
        # Process each test image
        correct = 0
        total_cer = 0.0
        inference_times = []
        valid_tests = 0
       
        with torch.no_grad():
            for i, img_path in enumerate(test_images, 1):
                # Load and preprocess
                img_tensor = preprocess_image(img_path, IMG_H, IMG_W)
                if img_tensor is None:
                    print(f'\n[{i}/{len(test_images)}] ⚠ Failed to load: {os.path.basename(img_path)}')
                    continue
               
                valid_tests += 1
               
                # Get ground truth
                gt_text = get_ground_truth(img_path)
               
                # Inference with timing
                img_tensor = img_tensor.to(device)
                start_inf = time.time()
                logits = model.forward_ctc(img_tensor) # [1, seq_len, num_classes]
                inference_times.append(time.time() - start_inf)
               
                probs = logits.exp()
                pred = probs.argmax(-1).cpu().numpy()[0] # [seq_len]
               
                # Decode prediction
                pred_text = decode_ctc_prediction(pred, ctc_idx_to_char)
               
                # Calculate CER
                error_rate = cer(gt_text, pred_text)
                total_cer += error_rate
               
                if error_rate == 0:
                    correct += 1
               
                # Display result
                status = '✓' if error_rate == 0 else ('~' if error_rate < 0.3 else '✗')
                print(f'\n[{i}/{len(test_images)}] {status} {os.path.basename(img_path)}')
                print(f' GT: "{gt_text}"')
                print(f' Pred: "{pred_text}"')
                print(f' CER: {error_rate:.4f}')
       
        # ====== 4. Summary ======
        print('\n' + '='*60)
        print('TESTING SUMMARY')
        print('='*60)
       
        if valid_tests > 0:
            print(f'Total images tested: {valid_tests}/{len(test_images)}')
            print(f'Correct predictions: {correct}/{valid_tests} ({100*correct/valid_tests:.1f}%)')
            print(f'Average CER: {total_cer/valid_tests:.4f}')
            print(f'Avg inference time: {np.mean(inference_times)*1000:.2f}ms/image')
        else:
            print('❌ No valid tests completed')
       
        print(f'\nModel: {MODEL_PATH}')
        print(f'Device: {device}')
        print('='*60)

TESTING MODEL ON 10 RANDOM IMAGES

Device: cuda
✓ Loading model from: /kaggle/working/sota_run/best_encoder.pth
✓ Model loaded successfully!
 Best epoch: 27
 Best ValCER: 0.0021

SAMPLING 10 RANDOM IMAGES (FAST & DIVERSE)
✓ Selected 10 diverse images in 14.532s via random walk

INFERENCE RESULTS

[1/10] ✓ 50_Soullessly_72886.jpg
 GT: "soullessly"
 Pred: "soullessly"
 CER: 0.0000

[2/10] ✓ 183_Trumpeting_80958.jpg
 GT: "trumpeting"
 Pred: "trumpeting"
 CER: 0.0000

[3/10] ✓ 114_WHATNOT_86133.jpg
 GT: "whatnot"
 Pred: "whatnot"
 CER: 0.0000

[4/10] ✓ 301_LUCE_45503.jpg
 GT: "luce"
 Pred: "luce"
 CER: 0.0000

[5/10] ✓ 469_Cottage_17339.jpg
 GT: "cottage"
 Pred: "cottage"
 CER: 0.0000

[6/10] ~ 393_detachment_21127.jpg
 GT: "detachment"
 Pred: "detachmert"
 CER: 0.1000

[7/10] ✓ 476_GAZES_31975.jpg
 GT: "gazes"
 Pred: "gazes"
 CER: 0.0000

[8/10] ✓ 17_koenig_42829.jpg
 GT: "koenig"
 Pred: "koenig"
 CER: 0.0000

[9/10] ✓ 147_Rechartered_63115.jpg
 GT: "rechartered"
 Pred: "rechartered"
 CER